# Silver Transformation
This notebook 
- flattens the raw `/5m` Bronze JSON into a clean, tabular fact table
- routes quality issues to a dead-letter queue
- flags completeness for future 'thin market' warnings
- merges idempotently into Delta.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, MapType,
    LongType, IntegerType, StringType
)
spark.sql("USE CATALOG osrs_pipeline") 
spark.sql("USE SCHEMA silver")


## Schema
Declaring `data` as a MapType instead of letting Spark infer a struct is what makes the item IDs map keys we can explode into rows, rather than over a thousand struct fields.

In [0]:
item_schema = StructType([
    StructField("avgHighPrice", LongType(), True),
    StructField("highPriceVolume", IntegerType(), True),
    StructField("avgLowPrice", LongType(), True),
    StructField("lowPriceVolume", IntegerType(), True),
])

bronze_schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("data", MapType(StringType(), item_schema), True),
])

## Read Bronze
Read all JSON files in the volume, with each JSON file being one 5 minute window with every item traded in the window

In [0]:

BRONZE_PATH = "/Volumes/osrs_pipeline/bronze/data_raw/*.json"
df = spark.read.schema(bronze_schema).json(BRONZE_PATH)

## Flatten
Each window comes in as a map of items, so I explode it out to one row per item and flatten
the nested price fields into columns. Item ID comes from the map key, and the timestamp sticks
to each item so nothing loses track of its window. I then renamed the columns to be more descriptive and changed from camel case to snake case.

In [0]:
exploded = df.select("timestamp", F.explode("data"))

silver = exploded.select(
    F.col("timestamp").alias("window_timestamp"),
    F.col("key").alias("item_id"),
    F.col("value.avgHighPrice").alias("avg_high_price"),
    F.col("value.highPriceVolume").alias("high_price_volume"),
    F.col("value.avgLowPrice").alias("avg_low_price"),
    F.col("value.lowPriceVolume").alias("low_price_volume"),
)

## Route: quality split
Every row goes to exactly one place: nothing gets silently dropped. Things that break the
API's contract (null or negative volume, missing item ID, a price that's zero or less) gets sent
to a dead-letter queue so I can inspect what went wrong. A null price is fine however as
that just means nothing traded on that side this window.

In [0]:
missing_key     = F.col("item_id").isNull()
null_volume     = F.col("high_price_volume").isNull() | F.col("low_price_volume").isNull()
bad_price       = (F.col("avg_high_price").isNotNull() & (F.col("avg_high_price") <= 0)) | \
                  (F.col("avg_low_price").isNotNull() & (F.col("avg_low_price") <= 0))
negative_volume = (F.col("high_price_volume") < 0) | (F.col("low_price_volume") < 0)

dlq_condition = missing_key | null_volume | bad_price | negative_volume

survivors = silver.filter(~dlq_condition)
dlq_df    = silver.filter(dlq_condition)

## Drop empty rows
If both prices are null, the row is of no use, so it gets dropped. This shouldn't
ever happen in the first place as items that haven't been traded in the window at all just don't show up in the response, but I guard in case the API ever changes how it behaves.

In [0]:
drop_condition    = F.col("avg_high_price").isNull() & F.col("avg_low_price").isNull()
survivors_cleaned = survivors.filter(~drop_condition)

## Flag completeness
Adds an `is_complete` flag = `true` if both sides have a price, `false` if one side is missing.
I keep the incomplete ones rather than dropping them as they could be useful later as a thin-market
signal (item trading on one side only). They just won't count toward flip margins since we need
both a buy and a sell price for that.

In [0]:
flagged = survivors_cleaned.withColumn(
    "is_complete",
    F.col("avg_high_price").isNotNull() & F.col("avg_low_price").isNotNull(),
)

In [0]:
# Add a human-readable UTC timestamp alongside the raw unix window_timestamp.
with_date = flagged.withColumn(
    "window_timestamp_utc", F.timestamp_seconds(F.col("window_timestamp"))
)

display(with_date)

## Persist to Silver: idempotent MERGE
The Delta Table gets created once with `CREATE TABLE IF NOT EXISTS` and then gets updated with a Delta `MERGE` keyed on `(item_id, window_timestamp)`.
That key allows this to be idempotent:

- **Same window again** → rows already exist, so they match and just update. This makes it safe to re-run.
- **New window** → nothing matches, so the rows insert as new history and the table grows over time.

The reason this works is that I key on the window's own timestamp, and not on when the pipeline actually
ran. A closed 5-minute window is frozen, so re-fetching it always gives me the same key and the
same data. A late run or a retry can't ever create duplicates.

In [0]:
with_date.createOrReplaceTempView("silver_updates")

spark.sql("""
CREATE TABLE IF NOT EXISTS silver_prices(
    window_timestamp BIGINT,
    item_id STRING,
    avg_high_price BIGINT,
    high_price_volume INT,
    avg_low_price BIGINT,
    low_price_volume INT,
    is_complete BOOLEAN,
    window_timestamp_utc TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql("""
MERGE INTO silver_prices target
USING silver_updates source
ON target.item_id = source.item_id AND target.window_timestamp = source.window_timestamp
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")